In [1]:
import json
import pickle
from collections import defaultdict

with open('../results/tracking_annotations.json', 'r') as f:
    data = json.load(f)

In [30]:
print(data['1'])

{'0': [{'bbox': [1723, 350, 1919, 865], 'track_id': 0, 'group_id': 0}], '1': [{'bbox': [1723, 351, 1919, 865], 'track_id': 0, 'group_id': 0}], '2': [{'bbox': [1715, 360, 1919, 862], 'track_id': 0, 'group_id': 0}], '3': [{'bbox': [1712, 354, 1919, 854], 'track_id': 0, 'group_id': 0}], '4': [{'bbox': [1710, 338, 1919, 848], 'track_id': 0, 'group_id': 0}], '5': [{'bbox': [1702, 330, 1919, 842], 'track_id': 0, 'group_id': 0}], '6': [{'bbox': [1695, 331, 1910, 835], 'track_id': 0, 'group_id': 0}], '7': [{'bbox': [1682, 317, 1899, 830], 'track_id': 0, 'group_id': 0}], '8': [{'bbox': [1675, 327, 1888, 827], 'track_id': 0, 'group_id': 0}], '9': [{'bbox': [1663, 317, 1872, 818], 'track_id': 0, 'group_id': 0}], '10': [{'bbox': [1653, 316, 1859, 815], 'track_id': 0, 'group_id': 0}], '11': [{'bbox': [1643, 316, 1846, 812], 'track_id': 0, 'group_id': 0}], '12': [{'bbox': [1636, 313, 1834, 813], 'track_id': 0, 'group_id': 0}], '13': [{'bbox': [1627, 318, 1824, 816], 'track_id': 0, 'group_id': 0}], '

In [44]:
import glob
import os

video_folders = glob.glob('../SEKAI_900_3/videos_frames/*/')
video_folders.sort()
#print(video_folders)

jsons_path = '../SEKAI_900_3/jsons_step1'

for folder_id, folder in enumerate(video_folders):
    clip = folder.split('/')[-2]
    json_file = f'{jsons_path}/{clip}.json'
    
    with open(json_file, 'r') as f:
        json_data_s1 = json.load(f)

    #print(json_file)
    for frame in range(50):

        #print(frame)
        current_frame_info={}
        current_frame_info['frame_id'] = frame+1
        current_frame_info['detections'] = []

        #print(data[str(folder_id+1)][str(frame)])
        bboxes = [group.get('bbox',) for group in data[str(folder_id+1)][str(frame)]]
        #print(len(bboxes))
        
        if bboxes[0] is None:
            json_data_s1['frames'].append(current_frame_info)
        else:
            for idx, box in enumerate(bboxes):
                current_frame_info['detections'].append({'track_id': idx+1, 'bbox':box})
            json_data_s1['frames'].append(current_frame_info)
        

    jsons_path2 = jsons_path.replace('jsons_step1','jsons_step2')
    os.makedirs(jsons_path2, exist_ok=True)
    with open(f"{json_file.replace('jsons_step1','jsons_step2')}", "w") as f:
        json.dump(json_data_s1, f, indent=4)
        

In [18]:
!pwd

/home/artcs1/web_annotation_tool_v1/notebooks


In [38]:
# READ SBU ANNOTATIONS to generate GT
import json
import pickle
from collections import defaultdict

with open('../results/tracking_annotations.json', 'r') as f:
    data = json.load(f)

counter = 15

all_results = {}
map_r = {1:0, 21:1, 41:2}


#results = {}

ann_frame = 1
#ann_frame = 21
#ann_frame = 41

for annotations in data:

    idx = int(annotations)
    bboxes = data[annotations]
    #idx = annotations['videoIndex']

    if idx not in all_results:
       all_results[idx-1] = {}
    
    
    #print(idx)
    #print(ann_frame)
    groups = []
    map_group_members = defaultdict(list)
    #print(bboxes['0'])
    for idx_group, group in enumerate(bboxes[str(ann_frame)]):
        if len(group)>0:
            idx_person = group['track_id']
            x1, y1, x2, y2 = group['bbox']
            group_label = group['group_id']
            map_group_members[group_label+1].append(idx_person+1)
            #print(group_label)
            
            dc  = 1
            lvl = 1
            GT_list = [idx-1, 1, int(x1), int(y1), int(x2), int(y2), group_label+1, dc, lvl]
            str_to_be_added = [str(k) for k in GT_list]
            str_to_be_added = (" ".join(str_to_be_added))
            f = open('../results/gt_gold_sekai_'+str(ann_frame+1)+'.txt', "a+")
            f.write(str_to_be_added + "\r\n")
            f.close()

    
    groups = list(map_group_members.values())
    all_results[idx-1][str(0)] = groups

#print(all_results[2])

#print(all_results)

if all_results:
    save_path = f"../results/gt_gold_sekai_{ann_frame+1}.pkl"
    with open(save_path, "wb") as f:
        pickle.dump(all_results[0], f)
